In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

print("--- APPROACH 1: UNRESTRICTED TREE (OVERFITTING) ---")
# Data loading & prep (same as File 1)
df = pd.read_csv('combined_stock_data.csv')
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df['High'] = df.groupby('Company Name')['High'].ffill()
df['Low'] = df.groupby('Company Name')['Low'].ffill()
df['Volume'] = df['Volume'].fillna(0)
df['Day_Range'] = df['High'] - df['Low']
df['Daily_Return'] = df.groupby('Company Name')['Close'].pct_change() * 100
df['Target_Trend'] = (df['Close'] > df['Open']).astype(int)

features = ['Open', 'High', 'Low', 'Volume', 'Day_Range', 'Daily_Return']
df_clean = df.dropna(subset=features + ['Target_Trend']).copy()
X = df_clean[features]
y = df_clean['Target_Trend']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# NO MAX DEPTH SET
dt_overfit = DecisionTreeClassifier(random_state=50)
dt_overfit.fit(X_train, y_train)

train_acc = accuracy_score(y_train, dt_overfit.predict(X_train))
test_acc = accuracy_score(y_test, dt_overfit.predict(X_test))

print(f"Training Accuracy: {train_acc * 100:.2f}% (It memorized the training data!)")
print(f"Testing Accuracy:  {test_acc * 100:.2f}% (But it performs worse on unseen data.)")

--- APPROACH 1: UNRESTRICTED TREE (OVERFITTING) ---
Training Accuracy: 100.00% (It memorized the training data!)
Testing Accuracy:  78.48% (But it performs worse on unseen data.)


In [2]:
from sklearn.tree import export_text

print("\n--- APPROACH 2: OPTIMIZED DEPTH & RULE EXTRACTION ---")
# Apply max_depth=10 to force generalization
dt_optimized = DecisionTreeClassifier(max_depth=10, random_state=50)
dt_optimized.fit(X_train, y_train)

opt_test_acc = accuracy_score(y_test, dt_optimized.predict(X_test))
print(f"Optimized Testing Accuracy: {opt_test_acc * 100:.2f}%\n")

print("--- WHITE BOX INTERPRETABILITY ---")
print("Because this is a Decision Tree, we can extract its exact trading logic:")
print(export_text(dt_optimized, feature_names=features, max_depth=2))


--- APPROACH 2: OPTIMIZED DEPTH & RULE EXTRACTION ---
Optimized Testing Accuracy: 84.42%

--- WHITE BOX INTERPRETABILITY ---
Because this is a Decision Tree, we can extract its exact trading logic:
|--- Daily_Return <= 0.20
|   |--- Daily_Return <= -0.34
|   |   |--- Daily_Return <= -0.79
|   |   |   |--- truncated branch of depth 8
|   |   |--- Daily_Return >  -0.79
|   |   |   |--- truncated branch of depth 8
|   |--- Daily_Return >  -0.34
|   |   |--- Daily_Return <= 0.01
|   |   |   |--- truncated branch of depth 8
|   |   |--- Daily_Return >  0.01
|   |   |   |--- truncated branch of depth 8
|--- Daily_Return >  0.20
|   |--- Daily_Return <= 0.72
|   |   |--- Daily_Return <= 0.41
|   |   |   |--- truncated branch of depth 8
|   |   |--- Daily_Return >  0.41
|   |   |   |--- truncated branch of depth 8
|   |--- Daily_Return >  0.72
|   |   |--- Daily_Return <= 1.15
|   |   |   |--- truncated branch of depth 8
|   |   |--- Daily_Return >  1.15
|   |   |   |--- truncated branch of d